# AI Solution Architecture — Applied

**AI Architecture · Week 19b**

Offline notebook for turning the insurance-underwriter assistant request into an architecture one-pager, ADR list, capacity estimate, and FDE hand-off checklist.

## 1. Discovery questions

Before drawing: Who are the users? Which workflow step hurts? Which documents are authoritative? What metadata gates access? What counts as success? Which regulations apply? What must be audited? What launch blockers exist?

## 2. Current-state versus future-state

Current: underwriter searches document store, checks Policy Admin, reads memos, asks senior reviewers, and manually writes audit notes.

Future:
```mermaid
flowchart LR
  U[Underwriter] --> Web[Existing web app AI panel]
  Web --> API[AI Assistant API]
  API --> PG[(pgvector chunks)]
  API --> AOAI[Azure OpenAI GPT-4o]
  API --> Audit[(Blob audit log)]
  API --> Review[Human review queue]
```


In [ ]:
from pydantic import BaseModel, Field
from dataclasses import dataclass

class Container(BaseModel):
    name: str
    technology: str
    responsibility: str

class ADR(BaseModel):
    title: str
    context: str
    decision: str
    consequences: str
    alternatives_rejected: list[str]

class Risk(BaseModel):
    failure_mode: str
    mitigation: str

class Architecture(BaseModel):
    title: str
    context: str
    containers: list[Container] = Field(default_factory=list)
    adrs: list[ADR] = Field(default_factory=list)
    risks: list[Risk] = Field(default_factory=list)

def render_one_pager(a: Architecture) -> str:
    lines = [f'# {a.title}', '', '## Context', a.context, '', '## Containers']
    lines += [f'- **{c.name}** ({c.technology}): {c.responsibility}' for c in a.containers]
    lines += ['', '## ADRs']
    lines += [f'- **{d.title}**: {d.decision}' for d in a.adrs]
    lines += ['', '## Risks']
    lines += [f'- **{r.failure_mode}** -> {r.mitigation}' for r in a.risks]
    return '\n'.join(lines)

arch = Architecture(
    title='Insurance Underwriting RAG Assistant',
    context='Embedded assistant answers policy, memo, and regulatory questions with citations, auditability, and Azure-only controls.',
    containers=[
        Container(name='Underwriter Web App', technology='existing app', responsibility='AI panel and feedback'),
        Container(name='AI Assistant API', technology='FastAPI or .NET', responsibility='orchestration, authZ, retrieval, streaming'),
        Container(name='Azure Postgres pgvector', technology='Postgres', responsibility='chunks, embeddings, metadata filters'),
        Container(name='Azure OpenAI GPT-4o', technology='managed model', responsibility='grounded answer generation'),
        Container(name='Blob Audit Log', technology='Azure Storage', responsibility='immutable prompt, chunks, answer, trace id'),
    ],
    adrs=[ADR(title='RAG first', context='docs change weekly', decision='Use RAG before fine-tuning', consequences='fresh citations, retrieval must be measured', alternatives_rejected=['fine-tune only'])],
    risks=[Risk(failure_mode='provider outage', mitigation='cached answers and human review'), Risk(failure_mode='retrieval regression', mitigation='canary eval gate')]
)
print(render_one_pager(arch))

## 3. Capacity and cost estimator

In [ ]:
@dataclass(frozen=True)
class Workload:
    users: int
    q_per_user_day: int
    prompt_tokens: int
    response_tokens: int
    prompt_price_1k: float
    completion_price_1k: float
    embedding_price_1k: float
    docs: int
    chunks_per_doc: int
    dims: int
    chunk_tokens: int = 800
    business_days: int = 22

def estimate(w: Workload):
    q_day = w.users * w.q_per_user_day
    q_month = q_day * w.business_days
    prompt_month = q_month * w.prompt_tokens
    completion_month = q_month * w.response_tokens
    llm_cost = prompt_month / 1000 * w.prompt_price_1k + completion_month / 1000 * w.completion_price_1k
    chunks = w.docs * w.chunks_per_doc
    storage_gb = chunks * w.dims * 4 / 1_000_000_000
    embedding_cost = chunks * w.chunk_tokens / 1000 * w.embedding_price_1k
    return {'q_day': q_day, 'q_month': q_month, 'prompt_month': prompt_month, 'completion_month': completion_month, 'llm_cost': llm_cost, 'storage_gb': storage_gb, 'embedding_cost': embedding_cost}

def show(label, w):
    r = estimate(w)
    print(label, f"q/day={r['q_day']:,}", f"monthly=${r['llm_cost']:,.2f}", f"vectors={r['storage_gb']:.2f}GB", f"embed=${r['embedding_cost']:.2f}")

base = Workload(50, 40, 1500, 800, 0.0025, 0.0100, 0.00002, 40_000, 10, 1536)
show('baseline', base)
show('500-user scale', Workload(**{**base.__dict__, 'users': 500}))

## 4. p99 latency budget

In [ ]:
latency = {'auth':100, 'retrieval_rerank':700, 'prompt_assembly':100, 'model_first_token':2000, 'stream_completion':4500, 'guardrails_audit':400, 'network_app':200}
print('p99 total ms', sum(latency.values()))
for stage, ms in latency.items():
    print(f'{stage:18s} {ms:4d} ms')

## 5. ADR list

In [ ]:
adrs = [
    ('RAG vs fine-tune', 'RAG first because documents change and citations matter.'),
    ('Azure OpenAI', 'Use Azure because identity, networking, monitoring, and procurement are already approved.'),
    ('pgvector', 'Use existing Azure Postgres for 400k chunks; revisit if scale or search quality demands it.'),
    ('LLMProvider port', 'Hide provider SDK behind a hexagonal port for tests and future portability.'),
    ('Streaming sync', 'Stream answers in-app with timeout; run ingestion asynchronously.'),
]
for title, decision in adrs:
    print(f'{title}: {decision}')

## 6. Failure-mode register

In [ ]:
failures = {
    'provider outage': 'cached-answers mode and human review fallback',
    'hallucination': 'groundedness threshold, refusal, escalation',
    'retrieval regression': 'canary corpus and eval gate before promotion',
    'PII leak': 'entitlement filters, private endpoints, redaction, audit',
    'cost blowup': 'per-user limits, budgets, cache, alerts',
    'stale regulatory feed': 'freshness monitor and visible source timestamp',
}
for mode, mitigation in failures.items():
    print(f'{mode} -> {mitigation}')

## 7. FDE hand-off checklist

In [ ]:
handoff = ['one-page C4 diagram', 'ADR bundle', 'capacity spreadsheet', 'risk register', 'evaluation plan', 'prompt registry seed', 'runbook', 'implementation backlog']
for i, item in enumerate(handoff, 1):
    print(f'{i}. {item}')

## Exercises

1. Add a business-unit entitlement field to retrieval filters.
2. Add a second capacity scenario with 20 percent cache hit rate.
3. Write an ADR rejecting direct model SDK calls inside route handlers.
4. Add launch acceptance criteria for groundedness, p99 latency, and audit completeness.

## Links
- Literature note: `02 Literature Notes/AI Architecture/AI Solution Architecture — Applied`
- Snippets: `04 Code Snippets/AI Architecture/AI Week 19b One Page Architecture Generator`, `.../AI Week 19b Capacity and Cost Estimator`
- MOC: `06 Maps of Content/AI Architecture Concepts`